# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E005b-perlabel-attention** — pooling A/B on the production
resnet34 @224 arm. One decode pass fills a *per-slice* feature bank; both arms
train from it: `mean_max` (the E003 control — must reproduce ~0.771, proving the
bank refactor is lossless) and `attention` (per-label gated attention MIL, the
experiment). Same folds/seed, combiner untouched, so the CV delta is attributable
to pooling alone. Requires the `WANDB_API_KEY` Kaggle secret.

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E005b — bump to the perlabel-attention squash merge before `kaggle kernels push`
# (this run needs HeadType/PerLabelAttentionHead and the per-slice bank, absent at b8d29f8).
COMMIT = "b8d29f8"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_blended import BLENDED_LABEL_SOURCE, train_heads_from_bank

In [ ]:
# Competition data (DICOMs + series metadata) is pre-mounted; the blended soft
# labels arrive via the attached private knee-labels dataset (issue #2 resolved).
from pathlib import Path

from knee.data import load_blended_labels

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

# Attached datasets moved mount points too; probe the plausible locations.
LABELS_CSV = "blended_labels_v1.csv"
label_candidates = [
    Path("/kaggle/input/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/josiemachalek/knee-labels") / LABELS_CSV,
    Path("/kaggle/input/datasets/knee-labels") / LABELS_CSV,
]
labels_path = next((p for p in label_candidates if p.exists()), None)
if labels_path is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"{LABELS_CSV} not found; mounts: {listing}")
labels = load_blended_labels(labels_path)
print(f"blended labels: {len(labels)} studies from {labels_path}")

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(
        project="rsna-knee",
        config={"commit": COMMIT, "label_source": BLENDED_LABEL_SOURCE, "n_label_studies": len(labels)},
    )
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E005b: production backbone (resnet34 @224, the E004-tie winner on efficiency);
# only the pooling changes between arms. The bank stores per-slice features, so
# pooling is a fit-time choice and both arms share one decode pass.
from knee.model import DEFAULT_BACKBONE, HeadType, KneeModel

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DEFAULT_BACKBONE
INPUT_SIZE = 224
ARMS = (HeadType.MEAN_MAX, HeadType.ATTENTION)  # control first

CHECKPOINT_DIR = Path("/kaggle/working")
BANK_PATH = CHECKPOINT_DIR / f"slice_bank_{BLENDED_LABEL_SOURCE}_resnet34.pt"

In [ ]:
# One threaded decode pass (the hours) fills the per-slice bank; each arm's head
# fits from cache (minutes). The attention arm gets its own model instance sharing
# the extractor's exact backbone weights — a checkpoint must pair the head with the
# backbone that produced its training features.
from knee.cv import collect_features, save_feature_bank

extractor = KneeModel(BACKBONE)  # also serves as the mean_max checkpoint carrier
bank = collect_features(COMP_ROOT, labels, series_types=SERIES_TYPES, model=extractor, input_size=INPUT_SIZE)
save_feature_bank(bank, BANK_PATH)
print("plane coverage:", {t.value: n for t, n in bank.plane_coverage().items()})

arm_results = {}
for head_type in ARMS:
    if head_type is extractor.head_type:
        arm_model = extractor
    else:
        arm_model = KneeModel(BACKBONE, head_type=head_type)
        arm_model.backbone.load_state_dict(extractor.backbone.state_dict())
    results = train_heads_from_bank(
        bank, CHECKPOINT_DIR / head_type.value, arm_model, input_size=INPUT_SIZE
    )
    arm_results[head_type] = results
    for result in results:
        print(f"{head_type.value}/{result.series_type.value}: trained on {result.n_studies} studies")

In [ ]:
# Local eval: pooled-OOF stratified CV per arm from the SAME bank, same folds/seed —
# the delta is attributable to pooling alone. Sanity gate: the mean_max control must
# land at ~0.771 (E003's number) or the bank refactor changed something and the
# attention number can't be trusted. Decision rule (experiments.md E005b): submit
# only if attention beats the control by more than the per-repeat spread — E004
# showed CV gains may not transfer, and a CV null isn't worth a submission.
from knee.cv import cross_validate

cvs = {}
for head_type in ARMS:
    print(f"=== {head_type.value} ===")
    cvs[head_type] = cv = cross_validate(bank, head_type=head_type)
    print(f"macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
          + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
    print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints land in per-arm subdirs (mean_max/, attention/) plus the bank, all
# persisted as notebook output. If the decision rule passes, publish the attention/
# subdir's three .pt files as a new knee-weights dataset VERSION that REPLACES the
# previous ones (inference rejects duplicate series types — never mount two sets).
import math

if run is not None:
    run.config.update({"backbone": BACKBONE, "input_size": INPUT_SIZE, "arms": [a.value for a in ARMS]})
    for head_type, results in arm_results.items():
        wandb.log(
            {
                f"in_sample_auc/{head_type.value}/{result.series_type.value}/{label}": auc
                for result in results
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    for head_type, cv in cvs.items():
        wandb.log(
            {
                f"cv/macro_auc/{head_type.value}": cv.macro_auc,
                **{f"cv/auc/{head_type.value}/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
            }
        )
    run.finish()